# Run mini-infer Pallas TPU kernels on a free Colab TPU

Needs only a Google account (no identity verification, no card).

1. **Runtime -> Change runtime type -> TPU**.
2. **Runtime -> Run all**.

The cell clones the public `main` branch and runs every kernel (dense,
paged decode, paged prefill, mixed prefill/decode; MHA + GQA) with
`interpret=False`, checking each against a NumPy reference. A real TPU run
prints `devices: [TpuDevice(...)]` and ends with `ALL PASS`; it refuses to
fall back to CPU, so green means it genuinely ran on the TPU.

Normally that is a single run, start to finish, with no installs. The cell
first probes the environment with a one-line Pallas kernel; only on a broken
`libtpu` (Colab's preinstalled `libtpu` older than its `jaxlib`, which rejects
compiled kernels with 'Unsupported version: expected <= 7 but got 8', or a TPU
VM whose `libtpu` failed to load at all) does it install the matched
`jax[tpu]` pair and restart the runtime once, because the bad `libtpu` is
already loaded in the process. **When it reconnects, run the cell again** and
it proceeds without reinstalling. On a non-TPU runtime it installs nothing and
just tells you to switch the runtime type.

To validate a feature branch instead, change the `BRANCH` variable at the
top of the cell.

In [ ]:
import os
import subprocess
import sys

import jax

print("jax", jax.__version__, "devices:", jax.devices())


def _mosaic_skew_error():
    """Compile one trivial Pallas kernel; return the error if libtpu rejects it.

    Colab images can ship a libtpu older than their jaxlib. The mismatch only
    surfaces when loading a compiled Mosaic module ("Unsupported version:
    expected <= 7 but got 8"), so probing costs one tiny kernel compile and
    nothing else. Any other error is a real bug and propagates.
    """
    import jax.numpy as jnp
    from jax.experimental import pallas as pl

    def _copy(x_ref, o_ref):
        o_ref[...] = x_ref[...]

    x = jnp.zeros((8, 128), jnp.float32)
    try:
        out = pl.pallas_call(_copy, out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype))(x)
        out.block_until_ready()
    except Exception as exc:
        msg = str(exc).lower()
        if "unsupported version" in msg or "serde version" in msg:
            return exc
        raise
    return None


_IS_TPU_VM = bool(os.environ.get("COLAB_TPU_1VM") or os.environ.get("TPU_ACCELERATOR_TYPE"))
_tpus = [d for d in jax.devices() if getattr(d, "platform", "") == "tpu"]
if _tpus:
    _libtpu_problem = _mosaic_skew_error()
elif _IS_TPU_VM:
    # A TPU VM whose jax cannot see the TPU is the same environment problem
    # (broken or missing libtpu), so it takes the same alignment path below.
    _libtpu_problem = RuntimeError(
        "this is a TPU VM but jax sees no TPU (libtpu failed to initialize)"
    )
else:
    raise SystemExit("No TPU. Runtime -> Change runtime type -> TPU, then run this cell again.")

_SENTINEL = "/content/.jax_tpu_aligned"
if _libtpu_problem is not None:
    if os.path.exists(_SENTINEL):
        raise SystemExit(
            "libtpu still broken after one alignment attempt:\n"
            f"{_libtpu_problem}\n"
            "Use the GCP Cloud TPU path in scripts/colab/README.md instead."
        )
    print(f"libtpu problem detected: {_libtpu_problem}")
    print("Installing the matched jax[tpu] pair (one time, ~155 MB).")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "jax[tpu]==" + jax.__version__,
            "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ],
        check=True,
    )
    open(_SENTINEL, "w").close()
    print(">>> Restarting the runtime so the new libtpu loads (the stale one is")
    print(">>> already in this process). Run this cell again when it reconnects. <<<")
    os.kill(os.getpid(), 9)  # the only way to unload the stale libtpu

DEST = "/content/mini-infer"
BRANCH = "main"
if os.path.isdir(DEST):
    # Colab VMs persist across runs; a clone-once bootstrap silently runs
    # stale code. Always sync the checkout to the branch head.
    subprocess.run(["git", "-C", DEST, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "https://github.com/JonathanBerhe/mini-infer.git",
            DEST,
        ],
        check=True,
    )
print(
    "running commit:",
    subprocess.run(
        ["git", "-C", DEST, "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip(),
)
sys.path.insert(0, os.path.join(DEST, "src"))
sys.path.insert(0, os.path.join(DEST, "scripts"))
os.chdir(DEST)

# Re-running the cell in a live runtime keeps old modules cached; purge ours
# so the freshly synced code is what actually runs.
_OURS = ("mini_infer", "run_tpu_pallas_kernels")
for _m in [m for m in list(sys.modules) if m.split(".")[0] in _OURS]:
    del sys.modules[_m]

import run_tpu_pallas_kernels as runner  # noqa: E402  (needs the sys.path setup above)

_rc = runner.main()
# Fail the cell on any parity FAIL so Run all cannot end green on a bad run.
assert _rc == 0, f"kernel validation FAILED (exit code {_rc})"
print("exit code:", _rc)